# Feature Engineering

Aggregates the review-level tables into one row per game: predictors from the early-access window only, outcomes from post-launch.

# Setup

In [1]:
import json
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

project_root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
clean_reviews_dir=project_root/'data'/'processed'/'interim'/'reviews'
toxicity_dir=project_root/'data'/'processed'/'interim'/'toxicity'
checkpoint_dir=project_root/'data'/'raw'/'_checkpoints'
processed_dir=project_root/'data'/'processed'
tables_dir=project_root/'data'/'processed'/'tables'
processed_dir.mkdir(parents=True,exist_ok=True)
tables_dir.mkdir(parents=True,exist_ok=True)

rejects_file_path=project_root/'data'/'processed'/'interim'/'clean_rejects.csv'
features_file_path=processed_dir/'game_features.parquet'

# every feature must add an entry here
data_dictionary=[]


def define(name,layer,definition,why):
    """register one feature in the dictionary. keeps the definition next to the code that
    computes it instead of in a document that drifts out of date"""
    data_dictionary.append({'feature':name,'layer':layer,'definition':definition,'why_it_is_here':why})
    return name


clean_games=sorted(int(f.stem) for f in clean_reviews_dir.glob('*.parquet'))
scored_games=sorted(int(f.stem) for f in toxicity_dir.glob('*.parquet'))

print('clean games  :',len(clean_games))
print('scored games :',len(scored_games))
print('checkpoints  :',len(list(checkpoint_dir.glob('*.json'))))

clean games  : 1793
scored games : 1793
checkpoints  : 1965


# The true review count, which is not the number of rows i have

Row counts are bounded by the collection cap, so volume comes from the count Steam reports.

In [2]:
def read_checkpoint(appid):
    """whatever the collector recorded about this game, or an empty dict"""
    checkpoint_file=checkpoint_dir/f'{appid}.json'
    if not checkpoint_file.exists():
        return {}
    try:
        return json.loads(checkpoint_file.read_text())
    except json.JSONDecodeError:
        # the collector may be mid write, which is a race rather than an error
        # itself on the next run
        return {}


checkpoint_summary=[]
for appid in clean_games:
    checkpoint=read_checkpoint(appid)
    checkpoint_summary.append({'appid':appid,
                               'has_checkpoint':bool(checkpoint),
                               'total_reviews_reported':checkpoint.get('total_reviews'),
                               'derived_launch_timestamp':checkpoint.get('derived_launch_timestamp'),
                               'cohort':checkpoint.get('cohort'),
                               'cap_was_hit':checkpoint.get('cap_was_hit')})

df_checkpoints=pd.DataFrame(checkpoint_summary)

print('WHAT THE COLLECTOR RECORDED')
print(df_checkpoints.to_string(index=False))
print(f'games with a checkpoint            : {df_checkpoints["has_checkpoint"].sum()} of {len(df_checkpoints)}')
print(f'games with a bisected launch date  : {df_checkpoints["derived_launch_timestamp"].notna().sum()}')
print()
print('any game without a checkpoint gets volume from its row count and is flagged.')
print('that flag has to stay visible - it is the difference between measuring a game')
print('and measuring my own collection budget')

WHAT THE COLLECTOR RECORDED
  appid  has_checkpoint  total_reviews_reported  derived_launch_timestamp      cohort  cap_was_hit
  25000            True                    9050              1507825710.0    launched        False
 108600            True                  466660                       NaN still_in_ea         True
 116100            True                     410              1401771844.0    launched        False
 200150            True                     377              1557259891.0    launched        False
 206500            True                   10857              1521307053.0    launched         True
 207140            True                   37129              1460190397.0    launched         True
 211820            True                  123211              1469177767.0    launched         True
 214420            True                    8341              1422442934.0    launched        False
 215550            True                     348              1478055383.0    laun

# The predictors, computed from the early access window only

Every predictor is computed from early-access rows, so nothing post-launch can reach the model.

In [3]:
post_launch_window_days=180


def split_cohorts(df_game,launch_timestamp):
    """early access rows and post launch rows, split on timestamp against the bisected
    launch date. never on the window label, never on the steam flag"""
    if launch_timestamp is None or pd.isna(launch_timestamp):
        # a game still in early access has no launch date
        # early access review and it has no outcome window at all
        return df_game,df_game.iloc[0:0]

    launch_timestamp=int(launch_timestamp)
    window_end=launch_timestamp+post_launch_window_days*86400
    df_ea=df_game[df_game['timestamp_created']<launch_timestamp]
    df_post=df_game[(df_game['timestamp_created']>=launch_timestamp)&
                    (df_game['timestamp_created']<window_end)]
    return df_ea,df_post


def summarise_early_access(df_ea,df_scores):
    """every predictor for one game, from its early access reviews only"""
    if df_ea.empty:
        return {}

    features={}
    ea_start=df_ea['review_date'].min()
    ea_end=df_ea['review_date'].max()
    ea_days=max((ea_end-ea_start).days,1)

    # post review persistence, the signature metric
    df_persist=df_ea[df_ea['has_playtime']&(~df_ea['persistence_is_negative'])]
    if not df_persist.empty:
        persistence=df_persist['post_review_persistence']

        features[define('pct_never_played_again','predictor',
                        'share of EA reviewers whose playtime_forever equals playtime_at_review '
                        'exactly - they never played again after reviewing',
                        'the churn signal itself. 22% of all rows corpus-wide. averaging this '
                        'into a mean persistence destroys it, so it gets its own column')]=\
            (persistence==0).mean()
        features[define('prp_median_hours','predictor',
                        'median of (playtime_forever - playtime_at_review) over EA reviewers, hours',
                        'the novel retention proxy. median not mean - the distribution is '
                        'violently right skewed and 22% of it sits at exactly zero')]=\
            persistence.median()/60
        features[define('prp_p75_hours','predictor',
                        '75th percentile of post-review persistence over EA reviewers, hours',
                        'keeps signal when the median is pinned at zero by the churn mass')]=\
            persistence.quantile(0.75)/60
        features[define('playtime_at_review_median_hours','predictor',
                        'median playtime_forever at the moment of reviewing, in hours',
                        'controls for prior investment - persistence means something different '
                        'for a 2 hour player than a 200 hour one')]=\
            df_persist['playtime_at_review'].median()/60

    # volume and velocity 
    features[define('ea_duration_days','predictor',
                    'days between first and last EA review',
                    'how long the game spent in early access - a confounder for every volume measure')]=ea_days
    features[define('n_ea_reviews_collected','covariate',
                    'EA rows actually collected for this game',
                    'the sample size behind every aggregate here. NOT a volume measure - it is '
                    'capped per window, so for popular games it is a collection budget')]=len(df_ea)
    features[define('ea_reviews_per_day_collected','predictor',
                    'n_ea_reviews_collected divided by ea_duration_days',
                    'activity rate. a lower bound where the cap binds, which is why '
                    'collection_completeness travels with it')]=len(df_ea)/ea_days

    # review score level and trend
    features[define('ea_positive_share','predictor',
                    'share of EA reviews with voted_up = true',
                    'THE baseline competitor - the one number a studio already has for free. '
                    'the model has to beat this or it has not earned its complexity')]=df_ea['voted_up'].mean()

    # second half against first half, which is more robust than a slope here
    # robust on a game with a few hundred reviews
    midpoint=ea_start+(ea_end-ea_start)/2
    first_half=df_ea[df_ea['review_date']<midpoint]
    second_half=df_ea[df_ea['review_date']>=midpoint]
    if len(first_half)>10 and len(second_half)>10:
        features[define('ea_positive_share_trend','predictor',
                        'ea_positive_share in the second half of EA minus the first half',
                        'direction of travel. a game souring during EA is the pattern the '
                        'score exists to catch early')]=\
            second_half['voted_up'].mean()-first_half['voted_up'].mean()

    #reviewer quality covariates
    features[define('ea_author_games_owned_median','covariate',
                    'median num_games_owned of EA reviewers',
                    'controls for reviewer type - a 900 game library is a different reviewer')]=\
        df_ea['author_games_owned'].median()
    features[define('ea_author_review_count_median','covariate',
                    'median num_reviews written by EA reviewers',
                    'the same control from the other direction')]=df_ea['author_review_count'].median()
    features[define('ea_share_edited','covariate',
                    'share of EA reviews edited after posting',
                    'edited reviews carry a recomputed EA flag - this quantifies that exposure '
                    'and explains the flag/timestamp disagreement')]=df_ea['was_edited'].mean()

    #community tone, from the sampled toxicity scores
    if df_scores is not None and not df_scores.empty:
        ea_scores=df_scores[df_scores['review_id'].isin(set(df_ea['review_id']))]
        if not ea_scores.empty:
            features[define('ea_toxicity_mean','predictor',
                            'mean Detoxify toxicity over sampled EA reviews',
                            'the level of hostility during early access')]=ea_scores['toxicity'].mean()
            features[define('ea_toxicity_share_above_half','predictor',
                            'share of sampled EA reviews scoring toxicity > 0.5',
                            'a small permanently hostile core matters more than the average, '
                            'and the mean hides it')]=(ea_scores['toxicity']>0.5).mean()
            features[define('ea_sentiment_mean','predictor',
                            'mean VADER compound over sampled EA reviews',
                            'tone independent of hostility - a review can be sad without being toxic')]=\
                ea_scores['sentiment_compound'].mean()
            features[define('ea_censored_rate','predictor',
                            'share of sampled EA reviews where Steam redacted profanity',
                            'a profanity measure that survives Steam censoring the text')]=\
                (ea_scores['masked_profanity_runs']>0).mean()
            features[define('n_ea_reviews_scored','covariate',
                            'sampled EA reviews carrying toxicity scores',
                            'the sample size behind every tone feature above')]=len(ea_scores)

            score_mid=ea_scores['review_date'].min()+(ea_scores['review_date'].max()-ea_scores['review_date'].min())/2
            early_scores=ea_scores[ea_scores['review_date']<score_mid]
            late_scores=ea_scores[ea_scores['review_date']>=score_mid]
            if len(early_scores)>10 and len(late_scores)>10:
                features[define('ea_toxicity_trend','predictor',
                                'mean toxicity in the second half of EA minus the first half',
                                'a community souring during EA is the early warning the score is for')]=\
                    late_scores['toxicity'].mean()-early_scores['toxicity'].mean()
    return features


print('early access predictor builder defined - cohort split is by TIMESTAMP')

early access predictor builder defined - cohort split is by TIMESTAMP


# The outcomes, and the circularity control

Early-access reviewers are excluded from the post-launch outcome set.

In [4]:
def summarise_post_launch(df_game,df_post,df_scores):
    """every outcome for one game, from the 180 days after its bisected launch date.
    df_post is passed in already split on timestamp - this function never re-derives the
    cohort, so there is exactly one definition of it in the notebook"""
    if df_post.empty:
        return {}

    features={}
    features[define('n_post_launch_reviews','outcome',
                    'reviews collected in the 180 days after the bisected launch date',
                    'the denominator behind every outcome below')]=len(df_post)
    features[define('post_positive_share','outcome',
                    'share of post-launch reviews with voted_up = true',
                    'RQ1 outcome - did the game land well with players after full release')]=\
        df_post['voted_up'].mean()

    df_post_persist=df_post[df_post['has_playtime']&(~df_post['persistence_is_negative'])]
    if not df_post_persist.empty:
        features[define('post_prp_median_hours','outcome',
                        'median post-review persistence of post-launch reviewers, in hours',
                        'RQ1 retention outcome, measured the same way as the EA predictor')]=\
            df_post_persist['post_review_persistence'].median()/60

    if df_scores is not None and not df_scores.empty:
        post_scores=df_scores[df_scores['review_id'].isin(set(df_post['review_id']))]
        if not post_scores.empty:
            features[define('post_toxicity_mean_uncontrolled','outcome',
                            'mean toxicity over all sampled post-launch reviews',
                            'RQ2 outcome without the cohort control - reported for comparison only')]=\
                post_scores['toxicity'].mean()

            # the rq2 control - drop anyone who also reviewed during early access
            ea_authors=set(df_game.loc[df_game['timestamp_created']<df_post['timestamp_created'].min(),
                                       'author_hash'])
            post_author_by_review=df_game.set_index('review_id')['author_hash']
            score_authors=post_scores['review_id'].map(post_author_by_review)
            not_an_ea_reviewer=~score_authors.isin(ea_authors)

            features[define('post_toxicity_mean','outcome',
                            'mean toxicity over sampled post-launch reviews, EXCLUDING anyone '
                            'who also reviewed during EA',
                            'RQ2 headline outcome - the circularity-controlled version')]=\
                post_scores.loc[not_an_ea_reviewer,'toxicity'].mean()
            features[define('n_excluded_by_cohort_control','outcome',
                            'sampled post-launch reviews removed by the EA-reviewer exclusion',
                            'the control removes almost nobody - reporting the number is the '
                            'honest version of applying it')]=int((~not_an_ea_reviewer).sum())
    return features


print('outcome builder defined')

outcome builder defined


# Building the table

In [5]:
feature_rows=[]

for appid in clean_games:
    df_game=pd.read_parquet(clean_reviews_dir/f'{appid}.parquet',
                            columns=['review_id','author_hash','timestamp_created','review_date',
                                     'written_during_early_access','voted_up','has_playtime',
                                     'persistence_is_negative','post_review_persistence',
                                     'playtime_at_review','author_games_owned',
                                     'author_review_count','was_edited'])
    scores_file=toxicity_dir/f'{appid}.parquet'
    df_scores=pd.read_parquet(scores_file) if scores_file.exists() else None
    checkpoint=read_checkpoint(appid)

    launch_timestamp=checkpoint.get('derived_launch_timestamp')
    df_ea,df_post=split_cohorts(df_game,launch_timestamp)

    row={'appid':appid}
    row.update(summarise_early_access(df_ea,df_scores))
    row.update(summarise_post_launch(df_game,df_post,df_scores))

    # 19% of games hit the cap, understating volume by a median 9.5x
    # exactly the popular titles. this ratio is how the model can see that
    total_reported=checkpoint.get('total_reviews')
    row[define('total_reviews_reported','covariate',
               'total review count Steam reports for the game via query_summary',
               'the only trustworthy volume measure. row counts are a collection budget, '
               'not a property of the game')]=total_reported
    row[define('collection_completeness','covariate',
               'rows collected divided by total_reviews_reported',
               'how much of the game I hold. the cap binds non-randomly on popular titles, '
               'so this has to be visible to the model rather than hidden')]=        (len(df_game)/total_reported if total_reported else None)
    row[define('flag_timestamp_disagreement','covariate',
               'share of rows where the Steam EA flag disagrees with the timestamp split',
               'quantifies the edited-review effect per game - the reason cohort membership '
               'is timestamp-based and only the text description is flag-based')]=        (df_game['written_during_early_access']!=(df_game['timestamp_created']<launch_timestamp)).mean()         if launch_timestamp else None

    # identity and split keys, including the bisected launch date
    # collector's bisect, never from the release date on the reviews
    row['launch_date']=pd.to_datetime(launch_timestamp,unit='s') if launch_timestamp else pd.NaT
    row['cohort']=checkpoint.get('cohort','unknown')
    row['volume_from_row_count']=total_reported is None
    row['cap_was_hit']=bool(checkpoint.get('cap_was_hit',False))
    row['has_toxicity_scores']=df_scores is not None and not (df_scores is None or df_scores.empty)
    feature_rows.append(row)

df_features=pd.DataFrame(feature_rows)

# english share comes from stage 2, measured before the filter ran
if rejects_file_path.exists():
    df_language=pd.read_csv(rejects_file_path)[['appid','english_share','n_reviews_all_languages']]
    df_features=df_features.merge(df_language,on='appid',how='left')

df_features.to_parquet(features_file_path,index=False)

print('GAME FEATURE TABLE')
print(f'games   : {len(df_features)}')
print(f'columns : {len(df_features.columns)}')
print()
print(df_features[['appid','cohort','n_ea_reviews_collected','ea_positive_share',
                   'prp_median_hours','ea_toxicity_mean','post_positive_share',
                   'post_toxicity_mean']].round(3).to_string(index=False))
print('written to',features_file_path)

GAME FEATURE TABLE
games   : 1793
columns : 35

  appid      cohort  n_ea_reviews_collected  ea_positive_share  prp_median_hours  ea_toxicity_mean  post_positive_share  post_toxicity_mean
  25000    launched                  2042.0              0.749             4.267             0.066                0.602               0.039
 108600 still_in_ea                  1496.0              0.871           123.517             0.060                  NaN                 NaN
 116100    launched                    40.0              0.875             0.992             0.024                0.529               0.046
 200150    launched                   163.0              0.423             0.700             0.045                1.000               0.008
 206500    launched                  2278.0              0.814             3.133             0.036                0.698               0.044
 207140    launched                  1640.0              0.965             7.683             0.038              

# The data dictionary

Written from the code that defines the features, so it cannot drift from them.

In [6]:
df_dictionary=pd.DataFrame(data_dictionary).drop_duplicates(subset='feature')
df_dictionary.to_csv(tables_dir/'data_dictionary.csv',index=False)

print('DATA DICTIONARY')
for layer in ['predictor','covariate','outcome']:
    subset=df_dictionary[df_dictionary['layer']==layer]
    print(f'\n--- {layer.upper()} ({len(subset)}) ---')
    for _,entry in subset.iterrows():
        print(f'  {entry["feature"]}')
        print(f'      is   : {entry["definition"]}')
        print(f'      why  : {entry["why_it_is_here"]}')
print(f'{len(df_dictionary)} features documented ->',tables_dir/'data_dictionary.csv')

DATA DICTIONARY

--- PREDICTOR (13) ---
  pct_never_played_again
      is   : share of EA reviewers whose playtime_forever equals playtime_at_review exactly - they never played again after reviewing
      why  : the churn signal itself. 22% of all rows corpus-wide. averaging this into a mean persistence destroys it, so it gets its own column
  prp_median_hours
      is   : median of (playtime_forever - playtime_at_review) over EA reviewers, hours
      why  : the novel retention proxy. median not mean - the distribution is violently right skewed and 22% of it sits at exactly zero
  prp_p75_hours
      is   : 75th percentile of post-review persistence over EA reviewers, hours
      why  : keeps signal when the median is pinned at zero by the churn mass
  playtime_at_review_median_hours
      is   : median playtime_forever at the moment of reviewing, in hours
      why  : controls for prior investment - persistence means something different for a 2 hour player than a 200 hour one
  ea_du

# The assertion cell

In [7]:
df_check=pd.read_parquet(features_file_path)

assert len(df_check)>0,'the feature table is empty'
assert df_check['appid'].is_unique,'more than one row for a game'

# every feature must still be on its natural scale
# normalised here instead of inside a fold
unscaled=['prp_median_hours','ea_duration_days','playtime_at_review_median_hours',
          'ea_author_games_owned_median']
for column in unscaled:
    if column in df_check and df_check[column].notna().any():
        assert df_check[column].dropna().max()>1.5,f'{column} looks scaled - scaling belongs in the model notebook'

# shares are shares
for column in ['ea_positive_share','post_positive_share','ea_toxicity_mean','post_toxicity_mean']:
    if column in df_check and df_check[column].notna().any():
        # dropna first, a still-in-EA game has a legitimate nan
        assert df_check[column].dropna().between(0,1).all(),f'{column} is outside 0 to 1'

# every documented feature exists, and every feature is documented
documented=set(pd.read_csv(tables_dir/'data_dictionary.csv')['feature'])
assert documented.issubset(df_check.columns),f'documented but missing: {documented-set(df_check.columns)}'

print(f'feature table passed: {len(df_check)} games, {len(df_check.columns)} columns')
print(f'{len(documented)} features documented, all present')
print(f'games with toxicity scores : {df_check["has_toxicity_scores"].sum()}')
print(f'games missing a checkpoint : {df_check["volume_from_row_count"].sum()} (volume is a row count, flagged)')

feature table passed: 1793 games, 35 columns
27 features documented, all present
games with toxicity scores : 1793
games missing a checkpoint : 0 (volume is a row count, flagged)
